# 🏛️ PRISM: Causal World Model Quickstart Tutorial

**PRISM** (*Provably Robust Interventions with Structural World Models*) implements Pearl's Causal Hierarchy (L1 Association $\to$ L2 Intervention $\to$ L3 Counterfactuals) for safety-critical physical systems.

This notebook guides you through:
1. **Initializing the PRISM Decision & Evidence Pipeline** with frozen weights (`baseline_005`).
2. **Auditing Upfront Model Trust & Telemetry Consistency** ($R_T$, $R_{8D}$, $D_{\text{lat}}$).
3. **Running Pearl Level-3 Twin-World Counterfactual Rollouts** ($do(A)$ under abducted exogenous noise).
4. **Evaluating Provable Safety Constraints & Headroom Margins** ($k=2$ uncertainty-adjusted bounds).
5. **Generating Tamper-Evident SHA-256 Audit Dossiers**.

In [ ]:
# Step 0: Imports & Environment Setup
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch

from prism.pipeline.engine import PrismPipeline, PrismPipelineConfig
from prism.cli import format_terminal_banner

# Verify PyTorch device and seeds
print(f"PyTorch version: {torch.__version__}")
torch.manual_seed(42)
np.random.seed(42)

## 1. Initialize the PRISM Pipeline Engine

We initialize `PrismPipeline` using the canonical calibrated checkpoint `artifacts/baseline_005`.

In [ ]:
config = PrismPipelineConfig(
    model_dir=Path("artifacts/baseline_005"),
    benchmark_dir=Path("data/decision_benchmark"),
    model_version="baseline_005",
)
pipeline = PrismPipeline(config=config)
print(f"✅ Successfully loaded PRISM world model from {config.model_dir}")

## 2. Execute End-to-End Decision on Scenario 1 (Normal Operations)

In Scenario 1, the plant operates within nominal limits. PRISM should verify high model trust, confirm safety headroom, and recommend `cand_do_nothing`.

In [ ]:
record_s1 = pipeline.run_scenario("scenario_01_do_nothing")
print(format_terminal_banner(record_s1))

## 3. Inspect Upfront Model Trust & Telemetry Diagnostics

Before trusting any downstream planner, PRISM runs an upfront audit:
- **$R_T$ (Core Temperature Residual):** $|\hat{T}_{\text{core}} - T_{\text{core}}|$
- **$R_{8D}$ (Multi-Channel Residual):** Normalized MAE across all 8 state dimensions
- **$D_{\text{lat}}$ (Latent Novelty):** Mahalanobis distance in the latent space $z$

In [ ]:
trust = record_s1.trust
print(f"Model Trust State:       {trust.trust_state}")
print(f"Reconstruction R_T:     {trust.reconstruction_residual_t_core:.2f} °C")
print(f"Reconstruction R_8D:    {trust.reconstruction_residual_8d:.4f}")
print(f"Latent Novelty D_lat:   {trust.latent_novelty_d:.2f} d_M")
print(f"Pass Trust Gate:        {trust.passed_trust_gate}")

## 4. Run Critical Scenario 3 (Coolant Pump Trip / Overheat Threat)

In Scenario 3, a coolant pump trip triggers rapid thermal accumulation. PRISM counterfactually evaluates intervention candidates, discovers that throttling power resolves the thermal runaway, and verifies safety constraints.

In [ ]:
record_s3 = pipeline.run_scenario("scenario_03_throttle")
print(format_terminal_banner(record_s3))

# Inspect the Pearl Level-3 Counterfactual Effect
if record_s3.counterfactual:
    cf = record_s3.counterfactual
    print("\n--- Twin-World Counterfactual Analysis ---")
    print(f"Intervention: do({cf.intervention.intervention_target} = {cf.intervention.counterfactual_value})")
    print(f"Causal Delta T_core (Peak): {cf.causal_effect.delta_t_core_peak:+.2f} °C")
    print(f"Causal Delta F_cool (Min):  {cf.causal_effect.delta_f_cool_min:+.2f} L/min")

## 5. Provable Safety Constraints & Headroom Margins

PRISM guarantees $k=2$ uncertainty-adjusted safety satisfaction ($95.4\%$ Gaussian coverage):
$$\hat{y}_{t} + 2\sigma_t \le y_{\text{limit}}$$

In [ ]:
safety = record_s3.safety
print(f"Safety State:           {safety.overall_state}")
print(f"Is Safe:                {safety.is_safe}")
lim = safety.limiting_constraint
print(f"Limiting Constraint:    {lim.constraint_name} ({lim.variable_symbol})")
print(f"Headroom Margin:        {lim.raw_margin:+.2f} {lim.unit}")
print(f"Uncertainty Penalty (k·σ): {lim.uncertainty_penalty:.2f} {lim.unit}")

## 6. Scenario 6: Epistemic Safeguard (Mandatory Abstention)

When sensors drift or physics unmodelability exceeds safety bounds, PRISM **abstiains** rather than issuing hallucinated control actions.

In [ ]:
record_s6 = pipeline.run_scenario("scenario_06_all_unsafe")
print(format_terminal_banner(record_s6))

print(f"Decision Status: {record_s6.decision.decision_status}")
print(f"Recommendation:  {record_s6.decision.recommendation}")
if record_s6.abstention:
    print(f"Abstention Root Cause: {record_s6.abstention.primary_trigger}")
    print(f"Epistemic Rationale:   {record_s6.abstention.explanation_text[:120]}...")

## 7. Cryptographic Tamper-Evident SHA-256 Provenance

Every PRISM decision produces a deterministic canonical hash linking inputs, model weights, trust residuals, and causal proof artifacts.

In [ ]:
for rec in [record_s1, record_s3, record_s6]:
    s_id = rec.provenance.scenario_id
    h = rec.provenance.unified_record_hash
    print(f"{s_id:<25} -> SHA-256: {h}")